# Production RAG Service — Notebook 2

## Retrieval and Answer Generation

Uses production ChromaDB, hybrid retrieval,
citation handling and Groq LLM client.

Uses a separate temporary vector database,
leaving the production collection untouched.

In [22]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent

assert (ROOT / "app").exists(), "Project root not found"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: D:\Agentic-Ai-Course\Porfolio1-production-rag-service


In [23]:
import tempfile
from uuid import uuid4

from app.ingestion.extractor import extract_document
from app.ingestion.chunker import create_chunks
from app.vector_store.store import VectorStoreService
from app.retrieval.retriever import RetrievalService
from app.generation.generator import GenerationService
from app.generation.llm_client import GroqLLMClient
from app.config import settings

print("Application imports PASSED")

Application imports PASSED


## 1. Index a Real PDF

Uses the TechNova manual.
The temporary database is not the application DB.

In [24]:
pdf = ROOT / "TechNova_HR_Policy_Manual.pdf"
assert pdf.exists(), f"Missing: {pdf}"

sections = extract_document(
    pdf.name, pdf.read_bytes()
)

chunks = create_chunks(
    document_id=uuid4(),
    filename=pdf.name,
    extracted_sections=sections
)

temp_dir = tempfile.TemporaryDirectory(
    prefix="rag_notebook_"
)
store = VectorStoreService(db_path=temp_dir.name)
count = store.add_chunks(chunks)

assert count == len(chunks)
assert store.count_chunks() == count

print("Indexed chunks:", count)
print("Temporary DB:", temp_dir.name)

2026-11-25 12:11:37,033 | INFO     | app.ingestion.extractor | PDF extraction completed: 20 text/table sections
2026-11-25 12:11:37,164 | INFO     | app.vector_store.store | ChromaDB initialized successfully
2026-11-25 12:11:37,164 | INFO     | app.ingestion.embeddings | Generating embeddings for 42 chunks
2026-11-25 12:11:41,939 | INFO     | app.ingestion.embeddings | Successfully embedded 42 chunks
2026-11-25 12:11:42,048 | INFO     | app.vector_store.store | Stored 42 chunks successfully
Indexed chunks: 42
Temporary DB: C:\Users\DELL\AppData\Local\Temp\rag_notebook_30t_xwh8


## 2. Hybrid Retrieval

Show the retrieved chunk count, similarity scores,
page numbers and source text.

In [25]:
question = "How many annual leave days are allowed?"

retriever = RetrievalService(store)
results = retriever.retrieve(
    question=question,
    top_k=5,
    threshold=0.30
)

print("Question:", question)
print("Retrieved chunk count:", len(results))
assert results, "No chunks passed retrieval threshold"

for index, item in enumerate(results, start=1):
    metadata = item["metadata"]
    print(
        f"Chunk S{index}",
        "File:", metadata.get("filename"),
        "Page:", metadata.get("page"),
        "Score:", round(item["similarity_score"], 4)
    )
    print(item["text"][:350], "\n")

2026-11-25 12:11:42,071 | INFO     | app.retrieval.retriever | Retrieval started | top_k=5 | threshold=0.30
2026-11-25 12:11:42,076 | INFO     | app.ingestion.embeddings | Generating query embedding
2026-11-25 12:11:42,145 | INFO     | app.ingestion.embeddings | Query embedding generated successfully
2026-11-25 12:11:42,155 | INFO     | app.vector_store.store | Hybrid retrieval completed | candidates=42 | returned=5
2026-11-25 12:11:42,156 | INFO     | app.retrieval.retriever | Retrieval completed | fetched=5 | relevant=5
Question: How many annual leave days are allowed?
Retrieved chunk count: 5
Chunk S1 File: TechNova_HR_Policy_Manual.pdf Page: 5 Score: 0.837
Table 1 Leave Type: Annual Leave | Annual Entitlement: 20 working days | Carry Forward: Max 10 days | Notes: Accrues from joining; 5+ days needs 2 weeks not Leave Type: Casual Leave | Annual Entitlement: 10 days | Carry Forward: Not allowed | Notes: Max 3 consecutive days Leave Type: Sick Leave | Annual Entitlement: 8 days | Carr

## 3. Grounded Groq Generation

Requires a configured GROQ_API_KEY.
This cell uses the actual production Groq client,
not a mock answer.

In [26]:
assert settings.GROQ_API_KEY.strip(), (
    "Missing GROQ_API_KEY in .env. "
    "Configure it before running this cell."
)

llm = GroqLLMClient()
try:
    generator = GenerationService(llm.generate)
    answer = await generator.generate_answer(
        question=question,
        chunks=results
    )
finally:
    await llm.close()

print("Question:", question)
print("Answer:", answer["answer"])
print("Citations:", answer["citations"])
print("Retrieved count:", answer["retrieved_count"])

assert "answer" in answer
assert "citations" in answer
assert answer["retrieved_count"] == len(results)

2026-11-25 12:11:42,218 | INFO     | app.generation.llm_client | Groq LLM client configured | model=openai/gpt-oss-20b
2026-11-25 12:11:42,218 | INFO     | app.generation.generator | Answer generation started | chunks=5
2026-11-25 12:11:42,227 | INFO     | app.generation.llm_client | Groq LLM request started
2026-11-25 12:11:42,828 | INFO     | app.generation.llm_client | Groq LLM request completed successfully
2026-11-25 12:11:42,828 | INFO     | app.generation.generator | Answer generated | citations=1
Question: How many annual leave days are allowed?
Answer: 20 working days
Citations: [{'source_id': 'S2', 'chunk_id': 'dbc90baa-6745-494d-b561-3aed09140e99-8', 'metadata': {'document_id': 'dbc90baa-6745-494d-b561-3aed09140e99', 'section': '3. Leave Policy', 'section_chunk_position': 0, 'page': 5, 'filename': 'TechNova_HR_Policy_Manual.pdf', 'chunk_position': 8}}]
Retrieved count: 5


## 4. Verify Refusal Without Evidence

No evidence means the generator must refuse
without making an LLM API call.

In [27]:
async def blocked_llm(messages):
    raise AssertionError(
        "LLM should not be called without evidence"
    )

refusal_generator = GenerationService(blocked_llm)
refusal = await refusal_generator.generate_answer(
    question="Question without retrieved evidence",
    chunks=[]
)

print(refusal)
assert refusal["citations"] == []
assert refusal["retrieved_count"] == 0
print("Refusal test PASSED")

2026-11-25 12:11:42,845 | INFO     | app.generation.generator | Answer refused: no retrieved chunks
{'answer': 'I could not find a supported answer in the uploaded documents.', 'citations': [], 'retrieved_count': 0}
Refusal test PASSED


## 5. Clean Up

Delete the temporary notebook database.

In [28]:
import gc

# ChromaDB may retain native database file handles on Windows.
# Do not delete the temporary database while the kernel is running.

if "retriever" in globals():
    del retriever

if "store" in globals():
    del store

gc.collect()

print("Temporary database path:", temp_dir.name)
print("Database cleanup deferred until kernel shutdown")
print("Notebook pipeline execution completed")

Temporary database path: C:\Users\DELL\AppData\Local\Temp\rag_notebook_30t_xwh8
Database cleanup deferred until kernel shutdown
Notebook pipeline execution completed
